# Similar pools: pair scores for tiers 3, 1 and 2 (GPU)

RUN_DECISIONS D20, post-hoc exploratory. Needs `similar_bundle.zip` in `My Drive/fragmentation_similar/` and the tier 3 model in `My Drive/fragmentation_t3/out/model/`. Re-running resumes: scored units are skipped. Each tier is its own cell; tier 2 is the slow one.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, zipfile, subprocess, json
BASE = '/content/drive/MyDrive/fragmentation_similar'
OUT = BASE + '/out'
T3MODEL = '/content/drive/MyDrive/fragmentation_t3/out/model'
assert os.path.exists(BASE + '/similar_bundle.zip'), 'similar_bundle.zip is not in ' + BASE
assert os.path.exists(T3MODEL + '/pair_model.safetensors'), 'tier 3 model is not in ' + T3MODEL
zipfile.ZipFile(BASE + '/similar_bundle.zip').extractall('/content/sim')
print(json.load(open('/content/sim/BUNDLE.json')))
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout or 'NO GPU')

In [ ]:
# tier 3
!cd /content/sim && python -u runs/similar_score_t3.py --model "$T3MODEL" --out "$OUT"

In [ ]:
# tier 1
!cd /content/sim && python -u runs/t12_score.py --tier T1 --out "$OUT"

In [ ]:
# tier 2 (slow)
!cd /content/sim && python -u runs/t12_score.py --tier T2 --out "$OUT"

In [ ]:
for t in ('T3', 'T1', 'T2'):
    p = f'{OUT}/{t}/SCORE_DONE.json'
    print(t, {k: v for k, v in json.load(open(p)).items() if k != 'manifest'} if os.path.exists(p) else 'NOT FINISHED')
print('SIMILAR SCORING SUMMARY PRINTED')